In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from start_line.plotting import *
from concept_abstraction.environments import Cyclic4StateEnv, TreeRepeatEnv
from concept_abstraction.training import train_model
from concept_abstraction.selection import greedy_selection
import torch
import torch.nn.functional as F


In [3]:
env = TreeRepeatEnv(concept_list=[4])
obs, _ = env.reset()

for _ in range(10):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    env.render()
    print(f"Action: {'left' if action == 0 else 'right'}, Reward: {reward}")

State: 3, Obs: [0]
Action: right, Reward: 0.1
State: 6, Obs: [0]
Action: left, Reward: 0.0
State: 13, Obs: [0]
Action: right, Reward: 0.1
State: 3, Obs: [0]
Action: left, Reward: 0
State: 7, Obs: [0]
Action: right, Reward: 0.1
State: 15, Obs: [0]
Action: right, Reward: 0.1
State: 3, Obs: [0]
Action: left, Reward: 0
State: 6, Obs: [0]
Action: left, Reward: 0.0
State: 12, Obs: [0]
Action: left, Reward: 0.0
State: 3, Obs: [0]
Action: right, Reward: 0.1


In [4]:
q_net = train_model(env)

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/training.py:34: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  torch.tensor(obs, dtype=torch.float32),


In [5]:
unique_obs = set()
state_to_obs = {}
obs_to_val = {}
obs_to_q = {}

# Get all unique observations and their value
for s in range(1,15):
    env.state = s
    o = tuple(env.get_observation())
    unique_obs.add(o)
    state_to_obs[s] = o

for o in unique_obs:
    o_tensor = torch.tensor(o).unsqueeze(0).float()
    q_vals = q_net(o_tensor).detach().squeeze().numpy()
    v = q_vals.max()
    obs_to_val[o] = v
    obs_to_q[o] = q_vals

vals = []
for s in range(1,15):
    o = state_to_obs[s]
    q_vals = obs_to_q[o]
    action_probs = F.softmax(torch.tensor(q_vals), dim=0).numpy()
    best_action = np.argmax(q_vals)
    print(f"State {s}, Observation {list(o)}, V(o) = {obs_to_val[o]:.3f}, Q = {q_vals}, π(o) = {action_probs.round(2)}, best_action = {best_action}")
    vals.append(obs_to_val[o])

State 1, Observation [1], V(o) = 4.222, Q = [4.2216234 2.6985722], π(o) = [0.82 0.18], best_action = 0
State 2, Observation [0], V(o) = 2.661, Q = [2.6613789 2.5654821], π(o) = [0.52 0.48], best_action = 0
State 3, Observation [0], V(o) = 2.661, Q = [2.6613789 2.5654821], π(o) = [0.52 0.48], best_action = 0
State 4, Observation [1], V(o) = 4.222, Q = [4.2216234 2.6985722], π(o) = [0.82 0.18], best_action = 0
State 5, Observation [0], V(o) = 2.661, Q = [2.6613789 2.5654821], π(o) = [0.52 0.48], best_action = 0
State 6, Observation [0], V(o) = 2.661, Q = [2.6613789 2.5654821], π(o) = [0.52 0.48], best_action = 0
State 7, Observation [0], V(o) = 2.661, Q = [2.6613789 2.5654821], π(o) = [0.52 0.48], best_action = 0
State 8, Observation [1], V(o) = 4.222, Q = [4.2216234 2.6985722], π(o) = [0.82 0.18], best_action = 0
State 9, Observation [0], V(o) = 2.661, Q = [2.6613789 2.5654821], π(o) = [0.52 0.48], best_action = 0
State 10, Observation [0], V(o) = 2.661, Q = [2.6613789 2.5654821], π(o) 

In [6]:
ground_truth_vals = []
for concepts in [[0,1,2,3],[4]]:
    print("\nConcepts: {}".format(concepts))
    env = TreeRepeatEnv(concept_list=concepts)
    q_net = train_model(env)

    unique_obs = set()
    state_to_obs = {}
    obs_to_val = {}
    obs_to_q = {}

    # Get all unique observations and their value
    for s in range(1,15):
        env.state = s
        o = tuple(env.get_observation())
        unique_obs.add(o)
        state_to_obs[s] = o

    for o in unique_obs:
        o_tensor = torch.tensor(o).unsqueeze(0).float()
        q_vals = q_net(o_tensor).detach().squeeze().numpy()
        v = q_vals.max()
        obs_to_val[o] = v
        obs_to_q[o] = q_vals

    vals = []
    for s in range(1,15):
        o = state_to_obs[s]
        q_vals = obs_to_q[o]
        action_probs = F.softmax(torch.tensor(q_vals), dim=0).numpy()
        best_action = np.argmax(q_vals)
        print(f"State {s}, Observation {list(o)}, V(o) = {obs_to_val[o]:.3f}, Q = {q_vals}, π(o) = {action_probs.round(2)}, best_action = {best_action}")
        vals.append(obs_to_val[o])
    
    if len(concepts) > 1:
        ground_truth_vals = vals 
    else:
        print("Value error: {}".format(np.max(np.abs(np.array(ground_truth_vals)-np.array(vals)))))


Concepts: [0, 1, 2, 3]


State 1, Observation [0, 0, 0, 1], V(o) = 10.072, Q = [10.071742   1.0108505], π(o) = [1. 0.], best_action = 0
State 2, Observation [0, 0, 1, 0], V(o) = 10.025, Q = [10.024882   1.0019273], π(o) = [1. 0.], best_action = 0
State 3, Observation [0, 0, 1, 1], V(o) = 1.022, Q = [0.931356  1.0216577], π(o) = [0.48 0.52], best_action = 1
State 4, Observation [0, 1, 0, 0], V(o) = 9.906, Q = [9.905715  0.9934701], π(o) = [1. 0.], best_action = 0
State 5, Observation [0, 1, 0, 1], V(o) = 1.023, Q = [0.9146073 1.0229206], π(o) = [0.47 0.53], best_action = 1
State 6, Observation [0, 1, 1, 0], V(o) = 1.005, Q = [0.90618235 1.0050472 ], π(o) = [0.48 0.52], best_action = 1
State 7, Observation [0, 1, 1, 1], V(o) = 1.019, Q = [0.9294917 1.0185087], π(o) = [0.48 0.52], best_action = 1
State 8, Observation [1, 0, 0, 0], V(o) = 9.956, Q = [9.956269 9.051411], π(o) = [0.71 0.29], best_action = 0
State 9, Observation [1, 0, 0, 1], V(o) = 1.014, Q = [0.9417076 1.0136766], π(o) = [0.48 0.52], best_action = 

In [56]:
env.concept_list

[4]

In [38]:
ground_truth_vals = []
for concepts in [[0,1,2], [0], [1], [2]]:
    print("\nConcepts: {}".format(concepts))
    env = Cyclic4StateEnv(concept_list=concepts)
    q_net = train_model(env)

    unique_obs = set()
    state_to_obs = {}
    obs_to_val = {}
    obs_to_q = {}

    # Get all unique observations and their value
    for s in range(4):
        env.state = s
        o = tuple(env.get_observation())
        unique_obs.add(o)
        state_to_obs[s] = o

    for o in unique_obs:
        o_tensor = torch.tensor(o).unsqueeze(0).float()
        q_vals = q_net(o_tensor).detach().squeeze().numpy()
        v = q_vals.max()
        obs_to_val[o] = v
        obs_to_q[o] = q_vals

    vals = []
    for s in range(4):
        o = state_to_obs[s]
        q_vals = obs_to_q[o]
        action_probs = F.softmax(torch.tensor(q_vals), dim=0).numpy()
        best_action = np.argmax(q_vals)
        print(f"State {s}, Observation {list(o)}, V(o) = {obs_to_val[o]:.3f}, Q = {q_vals}, π(o) = {action_probs.round(2)}, best_action = {best_action}")
        vals.append(obs_to_val[o])
    
    if len(concepts) > 1:
        ground_truth_vals = vals 
    else:
        print("Value error: {}".format(np.max(np.abs(np.array(ground_truth_vals)-np.array(vals)))))


Concepts: [0, 1, 2]
State 0, Observation [0, 1, 0], V(o) = 28.203, Q = [28.203487 28.189268 27.882391], π(o) = [0.37 0.36 0.27], best_action = 0
State 1, Observation [1, 0, 0], V(o) = 28.462, Q = [28.18836  28.247341 28.461842], π(o) = [0.3  0.31 0.39], best_action = 2
State 2, Observation [0, 0, 0], V(o) = 28.220, Q = [28.136944 28.220499 27.843828], π(o) = [0.35 0.38 0.26], best_action = 1
State 3, Observation [1, 1, 1], V(o) = 28.482, Q = [28.26284  28.2163   28.481514], π(o) = [0.31 0.3  0.39], best_action = 2

Concepts: [0]


KeyboardInterrupt: 

In [59]:
env = Cyclic4StateEnv(concept_list=[0])
obs, _ = env.reset()

for _ in range(10):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    env.render()
    print(f"Action: {'left' if action == 0 else 'right'}, Reward: {reward}")

Current state A, observation [0]
Action: right, Reward: 2.5
Current state A, observation [0]
Action: right, Reward: 2.5
Current state D, observation [1]
Action: left, Reward: 2.5
Current state D, observation [1]
Action: right, Reward: 2.85
Current state A, observation [0]
Action: right, Reward: 2.85
Current state D, observation [1]
Action: left, Reward: 2.5
Current state D, observation [1]
Action: right, Reward: 2.85
Current state D, observation [1]
Action: right, Reward: 2.85
Current state A, observation [0]
Action: right, Reward: 2.85
Current state D, observation [1]
Action: left, Reward: 2.5


In [61]:
ground_truth_vals = []
for concepts in [[0,1,2], [0], [1], [2]]:
    print("\nConcepts: {}".format(concepts))
    env = Cyclic4StateEnv(concept_list=concepts)
    q_net = train_model(env)

    unique_obs = set()
    state_to_obs = {}
    obs_to_val = {}
    obs_to_q = {}

    # Get all unique observations and their value
    for s in range(4):
        env.state = s
        o = tuple(env.get_observation())
        unique_obs.add(o)
        state_to_obs[s] = o

    for o in unique_obs:
        o_tensor = torch.tensor(o).unsqueeze(0).float()
        q_vals = q_net(o_tensor).detach().squeeze().numpy()
        v = q_vals.max()
        obs_to_val[o] = v
        obs_to_q[o] = q_vals

    vals = []
    for s in range(4):
        o = state_to_obs[s]
        q_vals = obs_to_q[o]
        action_probs = F.softmax(torch.tensor(q_vals), dim=0).numpy()
        best_action = np.argmax(q_vals)
        print(f"State {s}, Observation {list(o)}, V(o) = {obs_to_val[o]:.3f}, Q = {q_vals}, π(o) = {action_probs.round(2)}, best_action = {best_action}")
        vals.append(obs_to_val[o])
    
    if len(concepts) > 1:
        ground_truth_vals = vals 
    else:
        print("Value error: {}".format(np.max(np.abs(np.array(ground_truth_vals)-np.array(vals)))))


Concepts: [0, 1, 2]
State 0, Observation [0, 1, 0], V(o) = 28.149, Q = [28.148808 28.145075 27.840555], π(o) = [0.37 0.36 0.27], best_action = 0
State 1, Observation [1, 0, 0], V(o) = 28.483, Q = [28.212282 28.170809 28.483099], π(o) = [0.31 0.29 0.4 ], best_action = 2
State 2, Observation [0, 0, 0], V(o) = 28.160, Q = [28.070177 28.160076 27.858515], π(o) = [0.34 0.38 0.28], best_action = 1
State 3, Observation [1, 1, 1], V(o) = 28.490, Q = [28.176184 28.192835 28.489965], π(o) = [0.3 0.3 0.4], best_action = 2

Concepts: [0]
State 0, Observation [0], V(o) = 28.150, Q = [28.14999  28.149988 27.83499 ], π(o) = [0.37 0.37 0.27], best_action = 0
State 1, Observation [1], V(o) = 28.500, Q = [28.184992 28.18499  28.49999 ], π(o) = [0.3  0.3  0.41], best_action = 2
State 2, Observation [0], V(o) = 28.150, Q = [28.14999  28.149988 27.83499 ], π(o) = [0.37 0.37 0.27], best_action = 0
State 3, Observation [1], V(o) = 28.500, Q = [28.184992 28.18499  28.49999 ], π(o) = [0.3  0.3  0.41], best_ac

In [60]:
greedy_selection(env,1)

[0]

In [ ]:
# TODO: (1) Policy performance by concept(s) selected, (2) Approximation in concepts, (3) Algorithms for concept selection, (4) Human performance-based algorithms, (5) Human performance impact